In [100]:
import os
import torch
import pandas as pd
import numpy as np
print(torch.__version__)

2.11.0+cpu


In [101]:
from huggingface_hub import login
login()

Extraction Pipeline

In [102]:
from datasets import load_dataset
from itertools import islice
import json

def load_c4_dataset(configuration="realnewslike",
                    split="train",
                    streaming=True
    ):

    dataset = load_dataset("allenai/c4",configuration,split=split,streaming=streaming)
    return dataset


def process_c4(dataset,tokenizer,num_docs=1000,prompt_length=50,ref_length=200):
    samples = []
    seen_prompts = set()

    for i,item in enumerate(dataset):
        text = item["text"]
        token = tokenizer(text,add_special_token=False)["input_ids"]

        if len(token) < prompt_length + ref_length:
            continue

        prompt_tokens = token[:prompt_length]
        ref_tokens = token[prompt_length:prompt_length+ref_length]

        prompt_txt = tokenizer.decode(prompt_tokens,skip_special_tokens=True)
        ref_txt = tokenizer.decode(ref_tokens,skip_special_tokens=True)

        if prompt_txt in seen_prompts:
            continue

        seen_prompts.add(prompt_txt)

        data = {
            "prompt_text":prompt_txt,
            "prompt_tokens":prompt_tokens,
            "reference_text": ref_txt,
            "reference_tokens": ref_tokens,
            "full_text":text,
            "domain": "realnews"
        }
        samples.append(data)

        if len(samples) >= num_docs:
            break

    return samples

In [103]:
from transformers import AutoTokenizer

def get_tokenizer(model_name:str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return tokenizer

In [104]:
model_name="facebook/opt-2.7b"

In [ ]:
tokenizer = get_tokenizer(model_name)
dataset = load_c4_dataset(streaming=True)
samples = process_c4(dataset,tokenizer)

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/512 [00:00<?, ?it/s]

In [106]:
df = pd.DataFrame(samples)
df.to_csv("c4samples.csv",index=False)

In [108]:
df.to_parquet("c4_samples.parquet",index=False)

In [109]:
from google.colab import files

files.download("c4_samples.parquet")
files.download("c4samples.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [110]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [111]:
df.to_parquet(
    "/content/drive/MyDrive/c4_samples.parquet",
    index=False
)

df.to_csv(
    "/content/drive/MyDrive/c4_samples.csv",
    index=False
)

In [112]:
df.head()

,prompt_text,prompt_tokens,reference_text,reference_tokens,full_text,domain
0,"""Whoever gets him, they'll be getting a good o...","[2, 113, 39858, 1516, 123, 6, 51, 581, 28, 562...",\nIt’s an experience that might humble some. B...,"[50118, 243, 17, 27, 29, 41, 676, 14, 429, 140...","""Whoever gets him, they'll be getting a good o...",realnews
1,8i announced today it has launched a web playe...,"[2, 398, 118, 585, 452, 24, 34, 1660, 10, 3748...",walk around a person inside a virtual environ...,"[1656, 198, 10, 621, 1025, 10, 6229, 1737, 7, ...",8i announced today it has launched a web playe...,realnews
2,The Arlington County Board plans to vote Satur...,"[2, 133, 15728, 413, 1785, 708, 7, 900, 378, 1...","member board is expected to support the plan, ...","[8648, 792, 16, 421, 7, 323, 5, 563, 6, 61, 21...",The Arlington County Board plans to vote Satur...,realnews
3,The Hawaii man who was fired after issuing the...,"[2, 133, 6467, 313, 54, 21, 2277, 71, 10392, 5...",former state employee – a man in his 50s who ...,"[320, 194, 3200, 126, 10, 313, 11, 39, 654, 29...",The Hawaii man who was fired after issuing the...,realnews
4,Shania Twain expected to break the charts with...,"[2, 3609, 8295, 34878, 421, 7, 1108, 5, 12201,...",song LP was released on Sept. 29 and is set to...,"[17481, 8765, 21, 703, 15, 1919, 4, 1132, 8, 1...",Shania Twain expected to break the charts with...,realnews
